# prod_fraud_online_deploy / deploy

- Job ID `742346376031832`, run ID `755720320337683`, run result `SUCCESS`, run duration 598 s
- Task: `deploy` (task run `998477725909841`), result `SUCCESS`, execution 594 s
- Notebook: `/Workspace/Users/victor.rodrigues@databricks.com/tech_bar/databricks-tech-bar-fraud-project/prod/files/serving/notebooks/online_deploy`
- Parameters: `{}`
- Exported with `databricks jobs export-run <task_run_id> --views-to-export CODE` and converted to .ipynb with the cell outputs (text outputs only).

# 06 - Model Serving of the PIX fraud model (route-optimized + inference table)

Task `deploy` of the `fraud_online_deploy` job: publishes the `champion` version of the target `fraud_model`
(registered with the **feature spec**) to a **Model Serving endpoint**. On every request, the endpoint
automatically resolves the **6 batch features** (online lookup in the shared Lakebase) and the
on-demand `amount` (through `RequestSource`) — the client sends only `holder_id` + `created_at_brt` + `amount`.

## What this notebook does (idempotent)
1. **Deploy through the Databricks SDK** — `create_and_wait` the first time; `update_config_and_wait` +
   `put_ai_gateway` if the endpoint already exists (never recreates it destructively).
2. **Route optimization** (`route_optimized=True`) — low-latency route; set **only at
   creation**. Requires querying through `endpoint_url` (data plane) with an OAuth token.
3. **Inference table through AI Gateway** — `AiGatewayInferenceTableConfig` writes the payloads to the
   target catalog/schema (replaces the legacy `auto_capture_config`).
4. **Query permission** — grants `CAN_QUERY` on the endpoint to the viewers group (additive).

> `databricks-feature-lookup` is built into Model Serving — no need to package it.

In [ ]:
# values come from the job parameters (bundle); empty in standalone runs
dbutils.widgets.text("endpoint_name", "")   # endpoint name (<prefix>_fraud_serving)
dbutils.widgets.text("model_name", "")      # UC model packaged with the feature spec (catalog.schema.name)
dbutils.widgets.text("catalog", "")         # target catalog/schema: destination of the inference table
dbutils.widgets.text("schema", "")
dbutils.widgets.text("viewers_group", "")   # group that receives CAN_QUERY on the endpoint

CATALOG        = dbutils.widgets.get("catalog")
PROJECT_SCHEMA = dbutils.widgets.get("schema")
ENDPOINT_NAME  = dbutils.widgets.get("endpoint_name")
VIEWERS_GROUP  = dbutils.widgets.get("viewers_group")
SERVED_NAME    = "fraud-model"                     # name of the served entity in the endpoint
INFERENCE_TABLE_PREFIX = "fraud_serving_inference"   # AI Gateway -> <cat>.<schema>.<prefix>_payload
ALIAS          = "champion"

MODEL_FULL_NAME = dbutils.widgets.get("model_name")

## 1. Resolve the `champion` version of the model
The alias is resolved to a version number: the endpoint serves a fixed version, and a new deploy
follows the alias after each retraining.

In [ ]:
from mlflow.tracking import MlflowClient

mlflow_client = MlflowClient(registry_uri="databricks-uc")
model_version = mlflow_client.get_model_version_by_alias(MODEL_FULL_NAME, ALIAS).version
print(f"model={MODEL_FULL_NAME} | version {ALIAS}={model_version}")

model=vr_workspace.tech_bar.tech_bar_fraud_model | version champion=1


## 2. Endpoint config (served entity + traffic + AI Gateway inference table)
- **served entity**: UC model at the `champion` version, CPU/Small workload, scale-to-zero.
- **AI Gateway inference table**: writes the request/response payloads to the target UC
  (`<catalog>.<schema>.<prefix>_payload`).

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput, TrafficConfig, Route,
    ServingModelWorkloadType, AiGatewayConfig, AiGatewayInferenceTableConfig,
)

w = WorkspaceClient()

served_entity = ServedEntityInput(
    name=SERVED_NAME,
    entity_name=MODEL_FULL_NAME,
    entity_version=model_version,
    workload_type=ServingModelWorkloadType.CPU,
    workload_size="Small",
    scale_to_zero_enabled=True,
)
core_config = EndpointCoreConfigInput(
    name=ENDPOINT_NAME,
    served_entities=[served_entity],
    traffic_config=TrafficConfig(routes=[Route(served_model_name=SERVED_NAME, traffic_percentage=100)]),
)
ai_gateway = AiGatewayConfig(
    inference_table_config=AiGatewayInferenceTableConfig(
        enabled=True,
        catalog_name=CATALOG,
        schema_name=PROJECT_SCHEMA,
        table_name_prefix=INFERENCE_TABLE_PREFIX,
    )
)

## 3. Idempotent deploy
`create_and_wait` (with `route_optimized` + `ai_gateway`) only the first time. If it already exists, updates the
served config (`update_config_and_wait`) and the inference table (`put_ai_gateway`) — without recreating it.
`route_optimized` can only be set at creation (it does not change through update).

In [ ]:
from databricks.sdk.errors import ResourceDoesNotExist

try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"endpoint '{ENDPOINT_NAME}' exists (route_optimized={existing.route_optimized}) -> update")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[served_entity],
        traffic_config=core_config.traffic_config,
    )
    w.serving_endpoints.put_ai_gateway(
        name=ENDPOINT_NAME,
        inference_table_config=ai_gateway.inference_table_config,
    )
except ResourceDoesNotExist:
    print(f"endpoint '{ENDPOINT_NAME}' does not exist -> create (route_optimized=True)")
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=core_config,
        route_optimized=True,
        ai_gateway=ai_gateway,
    )

ep = w.serving_endpoints.get(ENDPOINT_NAME)
print(f"state={ep.state.ready} | config_update={ep.state.config_update} | route_optimized={ep.route_optimized}")
print(f"endpoint_url={ep.endpoint_url}")
print(f"ai_gateway.inference_table={ep.ai_gateway.inference_table_config if ep.ai_gateway else None}")

endpoint 'prod_fraud_serving' does not exist -> create (route_optimized=True)
state=EndpointStateReady.READY | config_update=EndpointStateConfigUpdate.NOT_UPDATING | route_optimized=True
endpoint_url=<data-plane-host>/7474655075961307/serving-endpoints/prod_fraud_serving/invocations
ai_gateway.inference_table=AiGatewayInferenceTableConfig(catalog_name='vr_workspace', enabled=True, schema_name='tech_bar', table_name_prefix='fraud_serving_inference')


## 4. Query permission for the viewers group
`update_permissions` **adds** `CAN_QUERY` to the group, keeping the other endpoint permissions
(`set_permissions` would replace the whole ACL).

In [ ]:
from databricks.sdk.service.serving import ServingEndpointAccessControlRequest, ServingEndpointPermissionLevel

acl = w.serving_endpoints.update_permissions(
    serving_endpoint_id=ep.id,
    access_control_list=[ServingEndpointAccessControlRequest(
        group_name=VIEWERS_GROUP, permission_level=ServingEndpointPermissionLevel.CAN_QUERY)])
for a in acl.access_control_list:
    print(a.user_name or a.group_name or a.service_principal_name, [p.permission_level.value for p in a.all_permissions])

6036593a-f1b6-49ee-aa6d-145161164de9 ['CAN_MANAGE']
ml-viewers ['CAN_QUERY']
admins ['CAN_MANAGE']


In [ ]:
import json

ep = w.serving_endpoints.get(ENDPOINT_NAME)
itc = ep.ai_gateway.inference_table_config if ep.ai_gateway else None
dbutils.notebook.exit(json.dumps({
    "endpoint_name":       ENDPOINT_NAME,
    "endpoint_id":         ep.id,
    "state_ready":         str(ep.state.ready),
    "config_update":       str(ep.state.config_update),
    "route_optimized":     ep.route_optimized,
    "endpoint_url":        ep.endpoint_url,
    "model_full_name":     MODEL_FULL_NAME,
    "model_version":       model_version,
    "inference_table":     f"{CATALOG}.{PROJECT_SCHEMA}.{INFERENCE_TABLE_PREFIX}_payload",
    "inference_enabled":   (itc.enabled if itc else None),
}))

{"endpoint_name": "prod_fraud_serving", "endpoint_id": "76b33e805f8641008c7541b8e43a1e2e", "state_ready": "EndpointStateReady.READY", "config_update": "EndpointStateConfigUpdate.NOT_UPDATING", "route_optimized": true, "endpoint_url": "<data-plane-host>/7474655075961307/serving-endpoints/prod_fraud_serving/invocations", "model_full_name": "vr_workspace.tech_bar.tech_bar_fraud_model", "model_version": "1", "inference_table": "vr_workspace.tech_bar.fraud_serving_inference_payload", "inference_enabled": true}